In [1]:
import glob
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv")


/tmp/ipykernel_16/2008091755.py:11: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


In [2]:
import glob
import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

In [3]:
TARGET = "TargetValue"
ID_COLUMN = "TransactionID"

# Keep IDs before removing them from model features
test_ids = test[ID_COLUMN].copy()

# ============================================================
# 2. COMBINE TRAIN AND TEST FOR CONSISTENT CATEGORY CODES
# This uses no target information.
# ============================================================

all_features = pd.concat(
    [
        train.drop(columns=[TARGET]),
        test
    ],
    axis=0,
    ignore_index=True
)

# ============================================================
# 3. DATE FEATURES
# ============================================================

transaction_date = pd.to_datetime(
    all_features["TransactionDate"],
    errors="coerce"
)

all_features["TransactionYear"] = transaction_date.dt.year
all_features["TransactionMonth"] = transaction_date.dt.month
all_features["TransactionQuarter"] = transaction_date.dt.quarter
all_features["TransactionDayOfWeek"] = transaction_date.dt.dayofweek
all_features["TransactionDayOfYear"] = transaction_date.dt.dayofyear

# Continuous representation of time
all_features["TransactionDays"] = (
    transaction_date - pd.Timestamp("1990-01-01")
).dt.days

# ============================================================
# 4. CLEAN MANUFACTURE YEAR AND CREATE ASSET AGE
# ============================================================

valid_year = all_features["ManufactureYear"].between(1900, 2020)

all_features["BadManufactureYear"] = (~valid_year).astype("int8")

all_features["ManufactureYearClean"] = (
    all_features["ManufactureYear"]
    .where(valid_year, np.nan)
)

all_features["AssetAge"] = (
    all_features["TransactionYear"]
    - all_features["ManufactureYearClean"]
)

# Remove impossible asset ages
all_features.loc[
    ~all_features["AssetAge"].between(0, 100),
    "AssetAge"
] = np.nan

# Drop the original dirty year
all_features = all_features.drop(columns=["ManufactureYear"])

# ============================================================
# 5. OPERATIONAL HOURS FEATURES
# ============================================================

all_features["HoursMissing"] = (
    all_features["OperationalHoursMeter"].isna().astype("int8")
)

all_features["LogOperationalHours"] = np.log1p(
    all_features["OperationalHoursMeter"].clip(lower=0)
)

# ============================================================
# 6. FREQUENCY FEATURES
# Useful for IDs and high-cardinality categories
# ============================================================

frequency_columns = [
    "AssetID",
    "ProductConfigID",
    "Spec_FullDescriptor",
    "Spec_BaseClass",
    "VendorPartnerID"
]

for column in frequency_columns:
    counts = all_features[column].value_counts(dropna=False)
    all_features[f"{column}_Frequency"] = (
        all_features[column].map(counts).astype("float32")
    )

# AssetID has extremely high cardinality and is usually almost unique.
# Keep its repetition frequency, but drop the raw ID to reduce overfitting.
all_features = all_features.drop(columns=["AssetID"])

# TransactionID is a record identifier, not a genuine measurement.
all_features = all_features.drop(columns=[ID_COLUMN])

# Original date string is no longer needed.
all_features = all_features.drop(columns=["TransactionDate"])

# ============================================================
# 7. MARK CATEGORICAL COLUMNS
# ============================================================

categorical_columns = (
    all_features.select_dtypes(include=["object"]).columns.tolist()
)

# ProductConfigID is an ID/category even though stored as an integer.
categorical_id_columns = [
    "ProductConfigID"
]

for column in categorical_id_columns:
    if column in all_features.columns:
        categorical_columns.append(column)

categorical_columns = list(dict.fromkeys(categorical_columns))

# Because train and test were combined first, their category definitions match.
for column in categorical_columns:
    all_features[column] = (
        all_features[column]
        .fillna("__MISSING__")
        .astype("category")
    )

# ============================================================
# 8. SPLIT BACK INTO TRAIN AND TEST
# ============================================================

X = all_features.iloc[:len(train)].copy()
X_test = all_features.iloc[len(train):].copy()

# Train on logarithmic target to align with RMSLE
y_log = np.log1p(train[TARGET])

# Stratification keeps cheap/medium/expensive machines represented
# similarly in training and validation sets.
target_bins = pd.qcut(
    y_log,
    q=20,
    labels=False,
    duplicates="drop"
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_log,
    test_size=0.20,
    random_state=42,
    stratify=target_bins
)

# ============================================================
# 9. LIGHTGBM PARAMETERS
# ============================================================

params = {
    "objective": "regression",
    "metric": "rmse",

    # Large maximum; early stopping chooses the useful number.
    "n_estimators": 3000,

    # Small steps generally generalize better.
    "learning_rate": 0.03,

    # Tree complexity.
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 40,

    # Row and feature sampling reduce overfitting.
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,

    # Regularization.
    "reg_alpha": 0.10,
    "reg_lambda": 1.00,

    # Categorical regularization.
    "cat_smooth": 20,
    "cat_l2": 10,

    # Histogram resolution.
    "max_bin": 255,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True
}

# ============================================================
# 10. VALIDATION MODEL
# ============================================================

model = lgb.LGBMRegressor(**params)

model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_columns,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=150,
            verbose=True
        ),
        lgb.log_evaluation(period=100)
    ]
)

# Convert logarithmic predictions back to dollar values
valid_predictions = np.expm1(
    model.predict(
        X_valid,
        num_iteration=model.best_iteration_
    )
)

valid_predictions = np.clip(valid_predictions, 0, None)

local_rmsle = np.sqrt(
    mean_squared_log_error(
        np.expm1(y_valid),
        valid_predictions
    )
)

print(f"\nBest iteration: {model.best_iteration_}")
print(f"Local validation RMSLE: {local_rmsle:.6f}")

# ============================================================
# 11. RETRAIN ON ALL TRAINING DATA
# ============================================================

# Slightly increase the selected number because the final model
# gets more training rows than the validation model.
final_iterations = max(
    100,
    int(model.best_iteration_ * 1.05)
)

final_params = params.copy()
final_params["n_estimators"] = final_iterations

final_model = lgb.LGBMRegressor(**final_params)

final_model.fit(
    X,
    y_log,
    categorical_feature=categorical_columns
)

# ============================================================
# 12. TEST PREDICTIONS AND SUBMISSION
# ============================================================

test_predictions = np.expm1(
    final_model.predict(X_test)
)

test_predictions = np.clip(test_predictions, 0, None)

submission[ID_COLUMN] = test_ids
submission[TARGET] = test_predictions

submission.to_csv("submission.csv", index=False)

print(f"Final iterations: {final_iterations}")
print("Created submission.csv")
print(submission.head())

Training until validation scores don't improve for 150 rounds
[100]	valid_0's rmse: 0.245852
[200]	valid_0's rmse: 0.222889
[300]	valid_0's rmse: 0.216174
[400]	valid_0's rmse: 0.213056
[500]	valid_0's rmse: 0.211266
[600]	valid_0's rmse: 0.210037
[700]	valid_0's rmse: 0.20916
[800]	valid_0's rmse: 0.208528
[900]	valid_0's rmse: 0.207959
[1000]	valid_0's rmse: 0.207454
[1100]	valid_0's rmse: 0.20711
[1200]	valid_0's rmse: 0.206776
[1300]	valid_0's rmse: 0.206493
[1400]	valid_0's rmse: 0.206313
[1500]	valid_0's rmse: 0.206126
[1600]	valid_0's rmse: 0.205932
[1700]	valid_0's rmse: 0.205815
[1800]	valid_0's rmse: 0.205664
[1900]	valid_0's rmse: 0.205537
[2000]	valid_0's rmse: 0.20542
[2100]	valid_0's rmse: 0.205332
[2200]	valid_0's rmse: 0.205248
[2300]	valid_0's rmse: 0.205122
[2400]	valid_0's rmse: 0.205031
[2500]	valid_0's rmse: 0.204965
[2600]	valid_0's rmse: 0.204921
[2700]	valid_0's rmse: 0.204868
[2800]	valid_0's rmse: 0.204786
[2900]	valid_0's rmse: 0.204722
[3000]	valid_0's rmse: